# Previsão de inadimplência e do valor em risco em clientes de cartão de crédito

**Projeto Final de Machine Learning Clássico**
**Tema 4.5: Default Credit Card, inadimplência e valor em risco**

Alunos: Otávio Bonini e Cauã Medeiros
Professor: Rodrigo Ramos Silva

**Base:** Default of Credit Card Clients (UCI), 30.000 clientes de um banco de Taiwan, abril a setembro de 2005.

**Pergunta:** quais clientes exigem ação preventiva de crédito e qual valor está exposto a esse risco?

**O que o projeto faz:** primeiro classifica quem tem risco de dar calote, depois estima quanto
dinheiro está em jogo com cada um, e no fim junta as duas coisas numa fila de prioridade.

## 1. Carregamento dos dados

In [ ]:
!pip install ucimlrepo

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=350)
X = dataset.data.features
y = dataset.data.targets

print(dataset.metadata['num_instances'], 'linhas')
print(dataset.metadata['num_features'], 'atributos')
print('valores ausentes:', dataset.metadata['has_missing_values'])

### 1.1 Renomeação das variáveis

A base vem com as colunas nomeadas de X1 a X23. Os nomes reais estão nos metadados.
Renomeamos também PAY_0 para PAY_1, porque a numeração original da série pula do 0 para o 2.

In [ ]:
df = pd.concat([X, y], axis=1)

df = df.rename(columns={
    'X1':  'LIMIT_BAL',
    'X2':  'SEX',
    'X3':  'EDUCATION',
    'X4':  'MARRIAGE',
    'X5':  'AGE',
    'X6':  'PAY_1',
    'X7':  'PAY_2',
    'X8':  'PAY_3',
    'X9':  'PAY_4',
    'X10': 'PAY_5',
    'X11': 'PAY_6',
    'X12': 'BILL_AMT1',
    'X13': 'BILL_AMT2',
    'X14': 'BILL_AMT3',
    'X15': 'BILL_AMT4',
    'X16': 'BILL_AMT5',
    'X17': 'BILL_AMT6',
    'X18': 'PAY_AMT1',
    'X19': 'PAY_AMT2',
    'X20': 'PAY_AMT3',
    'X21': 'PAY_AMT4',
    'X22': 'PAY_AMT5',
    'X23': 'PAY_AMT6',
    'Y':   'inadimplente',
})

print(df.shape)
df.head()

## 2. Análise Exploratória de Dados (EDA)

### 2.1 Quantos clientes deram calote

In [ ]:
# Proporção de inadimplentes na base
df['inadimplente'].value_counts(normalize=True)

### 2.2 Atraso de pagamento e calote

Para cada nível de atraso em setembro, qual a taxa de calote em outubro.
O count do lado mostra quantos clientes há em cada grupo. Grupos pequenos não são confiáveis.

In [ ]:
df.groupby('PAY_1')['inadimplente'].agg(['mean', 'count'])

### 2.3 Tipos de dados, valores nulos e duplicatas

In [ ]:
df.info()

print(f'Total de dados nulos: {df.isnull().sum().sum()}')
print(f'Linhas duplicadas: {df.duplicated().sum()}')

### 2.4 Estatísticas descritivas

As colunas de dinheiro são bem assimétricas. A média fica muito acima da mediana e o máximo é
dezenas de vezes maior. Não removemos esses valores extremos. O cliente com fatura alta não é
erro de digitação, é justamente o cliente de maior exposição, que o modelo precisa aprender a
tratar. Faturas negativas também são válidas: representam crédito a favor do cliente.

In [ ]:
df.describe().T

### 2.5 Categorias fora da documentação

A documentação da UCI descreve EDUCATION de 1 a 4 e MARRIAGE de 1 a 3.
Na prática aparecem outros códigos.

In [ ]:
print(df['EDUCATION'].value_counts().sort_index())
print()
print(df['MARRIAGE'].value_counts().sort_index())

### 2.6 Remoção dos registros inválidos

Optamos por remover os registros com códigos que não constam da documentação, por não ser
possível saber o que significam. Registramos abaixo quantas linhas saíram e o efeito na taxa
de inadimplência.

In [ ]:
antes = len(df)
df = df[~df['EDUCATION'].isin([0, 5, 6])]
df = df[df['MARRIAGE'] != 0]
df = df.copy()
print(f'Removidas {antes - len(df)} linhas ({(antes-len(df))/antes*100:.2f}%)')
print(f'Taxa de inadimplência após remoção: {df["inadimplente"].mean()*100:.2f}%')

### 2.7 Visualizações do diagnóstico

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1. Distribuição do alvo
df['inadimplente'].value_counts().plot.bar(ax=axes[0], color=['#4C72B0', '#C44E52'])
axes[0].set_title('Distribuição da inadimplência')
axes[0].set_xlabel('0 = adimplente | 1 = inadimplente')

# 2. Taxa de inadimplência por nível de atraso
taxa = df.groupby('PAY_1')['inadimplente'].mean() * 100
taxa.plot.bar(ax=axes[1], color='#C44E52')
axes[1].set_title('Taxa de inadimplência por atraso (PAY_1)')
axes[1].set_ylabel('% inadimplentes')

plt.tight_layout()
plt.show()

## 3. Pré-processamento e engenharia de atributos

### 3.1 Criação de atributos de comportamento

A informação de risco está espalhada em seis colunas mensais. Estas quatro colunas novas
resumem esse histórico em um número só.

In [ ]:
PAY  = [f'PAY_{i}' for i in range(1, 7)]
BILL = [f'BILL_AMT{i}' for i in range(1, 7)]
PAG  = [f'PAY_AMT{i}' for i in range(1, 7)]

# Em quantos dos 6 meses o cliente atrasou
df['n_atrasos'] = (df[PAY] >= 1).sum(axis=1)

# Pior atraso do período
df['max_atraso'] = df[PAY].max(axis=1)

# Quanto da fatura ele usa do limite disponível
df['util_media'] = df[BILL].mean(axis=1) / df['LIMIT_BAL']

# Quanto ele paga, em média
df['pgto_medio'] = df[PAG].mean(axis=1)

df[['n_atrasos', 'max_atraso', 'util_media', 'pgto_medio']].describe()

### 3.2 Separação entre atributos (X) e alvo (y)

In [ ]:
y = df['inadimplente']
X = df.drop(columns=['inadimplente'])
print(X.shape, y.shape)

### 3.3 Divisão entre treino e teste

Separamos 20% dos clientes num "cofre" que o modelo nunca vê durante o treino. É esse pedaço
que vai dizer se ele realmente aprendeu ou só decorou.

O stratify=y garante que os 22% de inadimplentes apareçam na mesma proporção nos dois lados.
O random_state=42 trava o sorteio, para o resultado ser sempre o mesmo.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Treino: {X_tr.shape[0]} ({y_tr.mean()*100:.2f}% inadimplentes)')
print(f'Teste:  {X_te.shape[0]} ({y_te.mean()*100:.2f}% inadimplentes)')

### 3.4 Padronização e codificação

**StandardScaler nas numéricas.** Coloca todas as colunas na mesma régua, com média 0 e desvio 1.
Sem isso, o limite de crédito, que chega a 1.000.000, atropelaria a idade, que vai só até 79.
Usamos o StandardScaler e não o MinMaxScaler porque as colunas de dinheiro têm valores extremos
e negativos, que fariam o MinMaxScaler espremer quase todo mundo perto do zero.

**One-Hot nas categóricas.** SEX, EDUCATION e MARRIAGE são rótulos, não quantidades. EDUCATION 3
não é o triplo de EDUCATION 1. O One-Hot quebra a coluna em várias de 0 e 1, evitando que o
modelo invente uma ordem que não existe.

As colunas PAY_ ficam como número de propósito, porque ali a ordem é real: 2 meses de atraso é
pior que 1 mês.

Aqui só montamos a receita. A transformação acontece dentro do Pipeline, na hora de treinar.
Assim a média e o desvio são calculados só com os dados de treino, sem vazar informação do teste.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

CATEGORICAS = ['SEX', 'EDUCATION', 'MARRIAGE']
NUMERICAS = [c for c in X.columns if c not in CATEGORICAS]

prep = ColumnTransformer([
    ('num', StandardScaler(), NUMERICAS),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAS),
])

print(f'{len(NUMERICAS)} numéricas para o StandardScaler')
print(f'{len(CATEGORICAS)} categóricas para o One-Hot')

## 4. Treinamento dos modelos de classificação

Treinamos dois modelos que se contrastam. A Regressão Logística é linear e devolve coeficientes
que dá para explicar, o que importa em crédito porque o banco precisa justificar a recusa ao
cliente. O Random Forest junta centenas de árvores e consegue pegar combinações entre variáveis,
então acerta mais, mas perde a explicação simples.

O class_weight='balanced' faz o erro sobre os inadimplentes pesar mais. Sem isso o modelo
descobre que dizer "ninguém dá calote" já acerta 78% e para de arriscar.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Modelo A: linear, fácil de explicar
log_reg = Pipeline([
    ('prep', prep),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
])

# Modelo B: conjunto de árvores, pega combinações
rf = Pipeline([
    ('prep', prep),
    ('clf', RandomForestClassifier(n_estimators=300, max_depth=16, min_samples_leaf=20,
                                   class_weight='balanced_subsample', n_jobs=-1, random_state=42))
])

log_reg.fit(X_tr, y_tr)
rf.fit(X_tr, y_tr)

print('Treinados.')

### 4.1 Avaliação no conjunto de teste

A métrica que decide é o F1, não a acurácia. Com 22% de inadimplentes, um modelo que não
sinaliza ninguém já acerta 78% e tem Recall zero.

Na matriz de confusão, o número de baixo à esquerda é o que importa: são os clientes que iam
dar calote e o modelo deixou passar.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

resultados = []
for nome, modelo in [('Regressão Logística', log_reg), ('Random Forest', rf)]:
    pred = modelo.predict(X_te)
    proba = modelo.predict_proba(X_te)[:, 1]
    resultados.append({
        'Modelo': nome,
        'Acurácia': accuracy_score(y_te, pred),
        'Precision': precision_score(y_te, pred),
        'Recall': recall_score(y_te, pred),
        'F1': f1_score(y_te, pred),
        'ROC-AUC': roc_auc_score(y_te, proba),
    })

tabela = pd.DataFrame(resultados).set_index('Modelo')
print((tabela * 100).round(2))

print()
print(confusion_matrix(y_te, rf.predict(X_te)))
print()
print(classification_report(y_te, rf.predict(X_te),
                            target_names=['Adimplente', 'Inadimplente']))

### 4.2 Ajuste dos hiperparâmetros

O GridSearchCV testa várias combinações de configuração e escolhe a melhor por validação
cruzada em 5 partes, usando só os dados de treino. O conjunto de teste continua fechado.

Esta célula demora alguns minutos.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_lr = GridSearchCV(
    Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=2000,
              class_weight='balanced', random_state=42))]),
    {'clf__C': [0.01, 0.1, 1.0]},
    scoring='f1', cv=cv, n_jobs=-1
)
grid_lr.fit(X_tr, y_tr)
print('LogReg:', grid_lr.best_params_, f'F1(CV) = {grid_lr.best_score_:.4f}')

grid_rf = GridSearchCV(
    Pipeline([('prep', prep), ('clf', RandomForestClassifier(n_estimators=200,
              class_weight='balanced_subsample', n_jobs=-1, random_state=42))]),
    {'clf__max_depth': [10, 16, None], 'clf__min_samples_leaf': [5, 20]},
    scoring='f1', cv=cv, n_jobs=-1
)
grid_rf.fit(X_tr, y_tr)
print('RF:', grid_rf.best_params_, f'F1(CV) = {grid_rf.best_score_:.4f}')

### 4.3 Resultado final

Tabela com os dois modelos já ajustados. É esta que vai para o relatório.

In [ ]:
melhores = {'Regressão Logística': grid_lr.best_estimator_,
            'Random Forest': grid_rf.best_estimator_}

resultados = []
for nome, modelo in melhores.items():
    pred = modelo.predict(X_te)
    proba = modelo.predict_proba(X_te)[:, 1]
    resultados.append({
        'Modelo': nome,
        'Acurácia': accuracy_score(y_te, pred),
        'Precision': precision_score(y_te, pred),
        'Recall': recall_score(y_te, pred),
        'F1': f1_score(y_te, pred),
        'ROC-AUC': roc_auc_score(y_te, proba),
    })

tabela = pd.DataFrame(resultados).set_index('Modelo')
print((tabela * 100).round(2))

## 5. Regressão: quanto dinheiro está em jogo

A classificação diz quem pode dar calote. Agora precisamos saber quanto cada cliente deve,
porque dois clientes com a mesma chance de calote não têm a mesma urgência se um deve 2 mil
e o outro deve 200 mil.

Usamos a fatura do mês mais recente (BILL_AMT1) como medida da exposição.

Cuidado importante: tiramos todas as colunas de setembro do conjunto de entrada. Se deixássemos
a fatura de setembro lá dentro, o modelo estaria "prevendo" uma coisa que ele já sabe. Sobram
20 colunas, todas do histórico de abril a agosto.

In [ ]:
# Alvo: a exposição financeira
y_reg = df['BILL_AMT1']

# Tudo que é de setembro sai, para não vazar informação
vazamento = ['BILL_AMT1', 'PAY_AMT1', 'PAY_1', 'inadimplente',
             'n_atrasos', 'max_atraso', 'util_media', 'pgto_medio']
X_reg = df.drop(columns=[c for c in vazamento if c in df.columns])

# Mesma divisão da classificação, para poder juntar as duas etapas depois
Xr_tr = X_reg.loc[X_tr.index]
Xr_te = X_reg.loc[X_te.index]
yr_tr = y_reg.loc[X_tr.index]
yr_te = y_reg.loc[X_te.index]

print('Colunas da regressão:', X_reg.shape[1])
print('Mesmos clientes nos dois testes:', (Xr_te.index == X_te.index).all())

### 5.1 Comparação dos modelos de regressão

A polinomial é a parte lenta desta célula.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CAT = ['SEX', 'EDUCATION', 'MARRIAGE']
NUM = [c for c in X_reg.columns if c not in CAT]

prep_reg = ColumnTransformer([
    ('num', StandardScaler(), NUM),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CAT),
])

modelos_reg = {
    'Linear Múltipla': Pipeline([
        ('prep', prep_reg),
        ('reg', LinearRegression())
    ]),
    'Polinomial (grau 2)': Pipeline([
        ('prep', prep_reg),
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('reg', Ridge(alpha=10.0))
    ]),
    'Random Forest Regressor': Pipeline([
        ('prep', prep_reg),
        ('reg', RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                      n_jobs=-1, random_state=42))
    ]),
}

linhas = []

for nome, pipe in modelos_reg.items():
    pipe.fit(Xr_tr, yr_tr)
    pred = pipe.predict(Xr_te)
    linhas.append({
        'Modelo': nome,
        'MAE': mean_absolute_error(yr_te, pred),
        'RMSE': mean_squared_error(yr_te, pred) ** 0.5,
        'R²': r2_score(yr_te, pred)
    })

tabela_reg = pd.DataFrame(linhas).set_index('Modelo').sort_values('RMSE')
print(tabela_reg.round(3))

## 6. Fatores de risco e valor em risco

### 6.1 Quais variáveis pesam mais

O Random Forest consegue dizer quanto cada coluna contribuiu para as decisões dele.

In [ ]:
import numpy as np

rf_final = grid_rf.best_estimator_

nomes = rf_final.named_steps['prep'].get_feature_names_out()
imp = pd.Series(rf_final.named_steps['clf'].feature_importances_,
                index=nomes).sort_values(ascending=False)

print('TOP 10 FATORES DE RISCO\n')
print(imp.head(10).round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
imp.head(15).sort_values().plot.barh(ax=ax, color='#4C72B0')
ax.set_title('Random Forest: importância dos atributos')
ax.set_xlabel('Importância')
plt.tight_layout()
plt.show()

### 6.2 Valor em Risco

Aqui juntamos as duas etapas. A classificação dá a probabilidade de calote e a regressão dá o
valor devido. Multiplicando os dois, cada cliente ganha um número em dinheiro que diz quanto
está realmente em risco com ele.

Ordenando por esse número, temos a fila de prioridade da equipe de cobrança.

In [ ]:
# Probabilidade de calote
p_default = rf_final.predict_proba(X_te)[:, 1]

# Exposição estimada, nos mesmos clientes
reg_final = modelos_reg['Random Forest Regressor']
exposicao = reg_final.predict(Xr_te)

# Valor em Risco = probabilidade x exposição
valor_risco = p_default * np.clip(exposicao, 0, None)

carteira = pd.DataFrame({
    'p_inadimplencia': p_default,
    'exposicao_estimada': exposicao,
    'exposicao_real': df.loc[X_te.index, 'BILL_AMT1'].values,
    'valor_em_risco': valor_risco,
    'inadimplente_real': y_te.values,
}, index=X_te.index).sort_values('valor_em_risco', ascending=False)

total_var = carteira['valor_em_risco'].sum()
total_exp = carteira['exposicao_real'].clip(lower=0).sum()
top10 = carteira.head(int(len(carteira) * 0.10))

print(f'Clientes na carteira de teste : {len(carteira)}')
print(f'Exposição total real          : NT$ {total_exp:,.0f}')
print(f'Valor em Risco total estimado : NT$ {total_var:,.0f}  ({total_var/total_exp*100:.1f}% da exposição)')
print()
print('OS 10% MAIS CRÍTICOS:')
print(f'  taxa real de inadimplência: {top10["inadimplente_real"].mean()*100:.1f}%  '
      f'(carteira inteira: {carteira["inadimplente_real"].mean()*100:.1f}%)')
print(f'  lift: {top10["inadimplente_real"].mean()/carteira["inadimplente_real"].mean():.2f}x')
print()
print('Top 10 clientes para ação preventiva:')
print(carteira.head(10).round(2).to_string())

### 6.3 Gráficos de avaliação

In [ ]:
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Curva ROC
for nome, modelo in [('Regressão Logística', grid_lr.best_estimator_),
                     ('Random Forest', rf_final)]:
    proba = modelo.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, proba)
    axes[0].plot(fpr, tpr, lw=2, label=f'{nome} (AUC={roc_auc_score(y_te, proba):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatório')
axes[0].set_xlabel('Falso Positivo'); axes[0].set_ylabel('Verdadeiro Positivo')
axes[0].set_title('Curva ROC'); axes[0].legend(fontsize=8)

# 2. Matriz de confusão
ConfusionMatrixDisplay.from_estimator(
    rf_final, X_te, y_te, ax=axes[1], cmap='Blues', colorbar=False,
    display_labels=['Adimplente', 'Inadimplente'])
axes[1].set_title('Matriz de confusão: Random Forest')

# 3. Distribuição do valor em risco
axes[2].hist(carteira['valor_em_risco'], bins=60, color='#C44E52')
axes[2].set_xlabel('Valor em Risco (NT$)'); axes[2].set_ylabel('Clientes')
axes[2].set_title('Distribuição do valor em risco')

plt.tight_layout()
plt.show()

## 7. Conclusão

### Melhor modelo de classificação

O Random Forest, com F1 de 0,534 e ROC-AUC de 0,780, contra 0,521 e 0,761 da Regressão
Logística. O F1 equilibra pegar os caloteiros e não incomodar clientes bons. O ROC-AUC mede se o
modelo coloca os clientes na ordem certa de risco, que é o que precisamos para montar a fila de
cobrança.

A Regressão Logística teve Recall um pouco maior porque, no corte padrão de 0,50, ela marca mais
clientes como arriscados. Por isso escolhemos pelo F1, que equilibra as duas coisas. A Logística
continua sendo uma boa opção quando o banco precisa explicar a decisão ao cliente.

### Melhor modelo de regressão

O Random Forest Regressor, com o menor erro médio (NT$ 7.566) e o maior R² (0,920), à frente da
Regressão Linear Múltipla e da Polinomial.

### Variáveis que mais pesam

PAY_1, max_atraso, n_atrasos, PAY_2, util_media e pgto_medio. Todas são de comportamento de
pagamento. Idade, sexo, escolaridade e estado civil ficam no fim da lista. Ou seja, o que o
cliente fez nos últimos meses explica o risco muito melhor que o perfil dele.

Quatro dessas seis colunas foram criadas por nós na engenharia de atributos.

### Recomendação para a política de crédito

Juntando as duas etapas, cada cliente ganha um valor em risco: a chance de calote vezes o valor
devido. O banco deve usar esse valor como fila de prioridade da cobrança, começando pelo topo.

Na carteira de teste, entre os 10% de clientes do topo da fila, 40,7% deram calote, contra 22,3%
na carteira toda. Ou seja, a fila encontra caloteiros quase duas vezes mais do que escolher ao
acaso. O modelo não substitui o analista: ele só organiza quem deve ser olhado primeiro.

### Limitações

A base é de Taiwan em 2005 e não serve para o Brasil de hoje sem novo treinamento. Seis meses de
histórico não pegam sazonalidade. O valor em risco é uma aproximação, não a perda contábil real,
porque não temos dados de recuperação nem de garantias. A base só tem clientes que já tiveram
crédito aprovado. Sexo, escolaridade e estado civil estão no modelo, e usar isso para decidir
crédito é questionável. E o modelo mostra associação, não causa.

### Ambiente de execução

Python 3 com pandas, numpy, scikit-learn, matplotlib, seaborn e ucimlrepo. Roda no Google Colab
sem GPU. Semente aleatória fixada em 42, então os números se repetem a cada execução.